# 04｜SQL 與資料倉儲（對應課程 M7）

**本 Notebook 的目標**

1. 用 Python 內建的 **SQLite** 建一個本機的「小型資料倉儲」，練習課程裡的 SQL
2. 親手寫出 M7 的月營收 SQL，並故意寫錯一版，看 JOIN 膨脹
3. 把**指標定義表**寫成 SQL，再和 pandas 的結果**對帳**
4. 把 CSV 轉成適合上傳 BigQuery 的格式（去掉 BOM、改表名）

> **為什麼用 SQLite？** 它內建在 Python 裡，不用帳號也不用網路。SQL 的觀念和 BigQuery 相同，
> 只有少數函式寫法不同（最後有對照表）。

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import islemart_lib as L

pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.4f}".format)
T = L.load_all()

## 1. 建立本機資料庫，載入 9 張表

表名去掉 `islemart_` 與 `_dataset`，和 M7 講義在 BigQuery 的命名一致。

**價格異常的標記**：SQLite 沒有中位數函式，所以先在 pandas 算好 `is_price_outlier` 欄位再載入。
（BigQuery 可以用 `APPROX_QUANTILES` 在 SQL 裡算，見本頁最後。）

In [ ]:
DB = Path("islemart.db")
DB.unlink(missing_ok=True)
con = sqlite3.connect(DB)

items = T["order_items"].copy()
items["is_price_outlier"] = L.flag_price_outliers(items, T["products"]).astype(int)

to_load = {**T, "order_items": items}
for name, df in to_load.items():
    df.to_sql(name, con, index=False)

def q(sql: str) -> pd.DataFrame:
    # 執行 SQL，回傳 DataFrame
    return pd.read_sql_query(sql, con)

q("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")

## 2. 載入後先對帳：列數一致嗎？

這是 M7 步驟 4。一段 SQL 列出所有表的列數，和原始 CSV 比較。

In [ ]:
counts_sql = " UNION ALL ".join(f"SELECT '{n}' AS 表名, COUNT(*) AS 資料庫列數 FROM {n}" for n in T)
counts = q(counts_sql)
counts["CSV 列數"] = counts["表名"].map({k: len(v) for k, v in T.items()})
counts["一致"] = counts["資料庫列數"] == counts["CSV 列數"]
assert counts["一致"].all()
counts

## 3. 月營收：正確版（先彙總，再合併）

和講義 M7 的 SQL 相同，只把 BigQuery 的 `FORMAT_DATE` 換成 SQLite 的 `strftime`，並補上異常值排除。

In [ ]:
monthly_sql = '''
WITH items AS (                              -- 1. 先把明細彙總成一列一筆訂單
    SELECT order_id,
           SUM(CASE WHEN is_price_outlier = 1 THEN 0 ELSE price END) AS item_total
    FROM order_items
    GROUP BY order_id
)
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
       COUNT(*)          AS orders,
       SUM(i.item_total) AS revenue
FROM orders AS o
JOIN items  AS i USING (order_id)            -- 2. 一對一合併，列數不會膨脹
WHERE o.order_status = 'delivered'           -- 3. 篩選條件來自指標定義
GROUP BY month
ORDER BY month
'''
monthly = q(monthly_sql)
monthly.tail(6)

## 4. 故意寫錯：直接 JOIN 三張表

把 orders、order_items、order_payments 直接合併再加總，看營收會多出多少。

In [ ]:
wrong_sql = '''
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
       SUM(CASE WHEN it.is_price_outlier = 1 THEN 0 ELSE it.price END) AS revenue_wrong
FROM orders AS o
JOIN order_items    AS it USING (order_id)
JOIN order_payments AS p  USING (order_id)   -- 一對多再一對多，列數膨脹
WHERE o.order_status = 'delivered'
GROUP BY month
'''
cmp = monthly.merge(q(wrong_sql), on="month")
cmp["多算"] = cmp["revenue_wrong"] - cmp["revenue"]
print(f"兩年合計多算：{cmp['多算'].sum():,.0f} 元")
cmp.tail(4)

## 5. 把指標定義表寫成 SQL

對照講義 M7 的指標定義表。每個指標一段 SQL，放在同一個字典裡，就是一份「可執行的指標定義表」。

In [ ]:
METRICS = {
"訂單數": '''
    SELECT COUNT(*) FROM orders
''',
"營收": '''
    SELECT SUM(CASE WHEN it.is_price_outlier = 1 THEN 0 ELSE it.price END)
    FROM order_items it JOIN orders o USING (order_id)
    WHERE o.order_status = 'delivered'
''',
"延遲率": '''
    SELECT AVG(CASE WHEN date(order_delivered_customer_date) > date(order_estimated_delivery_date)
                    THEN 1.0 ELSE 0.0 END)
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_delivered_customer_date >= order_delivered_carrier_date
''',
"回購率": '''
    WITH per_person AS (
        SELECT c.customer_unique_id, COUNT(*) AS n
        FROM orders o JOIN customers c USING (customer_id)
        GROUP BY c.customer_unique_id
    )
    SELECT AVG(CASE WHEN n >= 2 THEN 1.0 ELSE 0.0 END) FROM per_person
''',
"平均評分": '''
    WITH dedup AS (
        SELECT DISTINCT * FROM order_reviews                  -- 刪掉完全重複的列
    ), ranked AS (
        SELECT order_id, review_score,
               ROW_NUMBER() OVER (PARTITION BY order_id
                                  ORDER BY review_answer_timestamp DESC) AS rn
        FROM dedup
    )
    SELECT AVG(review_score) FROM ranked WHERE rn = 1          -- 每筆訂單只留一則
''',
}

sql_result = {name: con.execute(sql).fetchone()[0] for name, sql in METRICS.items()}
pd.Series(sql_result)

## 6. 對帳：SQL 和 pandas 算出來一樣嗎？

pandas 的數字來自 `islemart_lib` 的同一套定義。兩邊獨立計算，結果必須一致。

In [ ]:
wide = L.build_orders_wide(T)
per_person = wide.groupby("customer_unique_id").size()
pandas_result = {
    "訂單數": len(wide),
    "營收": L.revenue(wide),
    "延遲率": L.late_rate(wide),
    "回購率": (per_person >= 2).mean(),
    "平均評分": L.avg_score(wide),
}
recon = pd.DataFrame({"SQL": sql_result, "pandas": pandas_result})
recon["一致"] = (recon["SQL"] - recon["pandas"]).abs() < 1e-6
assert recon["一致"].all(), "SQL 與 pandas 不一致！"
recon

## 7. 準備上傳 BigQuery 的檔案

M7 的常見問題：CSV 含 BOM 時，BigQuery 第一個欄位名稱會出現怪字元。
這裡把每張表另存成**不含 BOM 的 UTF-8**，並改成短表名，放進 `bigquery_upload/`。

In [ ]:
out = Path("bigquery_upload")
out.mkdir(exist_ok=True)
for name, df in T.items():
    df.to_csv(out / f"{name}.csv", index=False, encoding="utf-8")   # utf-8 不含 BOM
sorted(p.name for p in out.glob("*.csv"))

### 用 bq 指令一次上傳（選用）

已安裝 Google Cloud SDK 的話，可以在終端機執行（`你的專案` 換成自己的專案 ID）：

```bash
bq mk --location=asia-east1 你的專案:islemart
for f in bigquery_upload/*.csv; do
  bq load --autodetect --source_format=CSV --skip_leading_rows=1 \
     "你的專案:islemart.$(basename "$f" .csv)" "$f"
done
```

不熟指令的話，用 BigQuery 網頁介面逐一上傳也可以（講義 M7 的五個步驟）。

## 8. SQLite 與 BigQuery 的寫法差異

| 用途 | SQLite | BigQuery |
| --- | --- | --- |
| 日期轉年月 | `strftime('%Y-%m', ts)` | `FORMAT_DATE('%Y-%m', DATE(ts))` |
| 取日期 | `date(ts)` | `DATE(ts)` |
| 日期相減 | `julianday(a) - julianday(b)` | `DATE_DIFF(a, b, DAY)` |
| 中位數 | 沒有內建 | `APPROX_QUANTILES(x, 2)[OFFSET(1)]` |
| 表名 | `orders` | `專案.islemart.orders` |

BigQuery 版本的價格異常標記，可以直接在 SQL 裡完成：

```sql
WITH cat_median AS (
  SELECT p.product_category_name AS cat,
         APPROX_QUANTILES(it.price, 2)[OFFSET(1)] AS median_price
  FROM islemart.order_items it
  JOIN islemart.products p USING (product_id)
  GROUP BY cat
)
SELECT it.*, it.price >= 20 * m.median_price AS is_price_outlier
FROM islemart.order_items it
JOIN islemart.products p USING (product_id)
JOIN cat_median m ON m.cat = p.product_category_name
```

In [ ]:
con.close()

## 小結

| 課堂上的 AI 做法 | 本 Notebook 的傳統做法 |
| --- | --- |
| 請 AI 寫 SQL 並逐句加註解 | 自己寫 SQL，用 `WITH` 分段，每段一個目的 |
| 指標定義表放進 Project | 指標定義寫成 `METRICS` 字典，可直接執行 |
| 和 Day 1 結果對帳 | SQL 與 pandas 兩邊獨立計算，`assert` 比對 |

**練習**：在 `METRICS` 加一個「平均客單價」（營收 ÷ 已送達訂單數），並用 pandas 對帳。